In [19]:
import os
from dotenv import load_dotenv
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated
from langchain_core.messages import BaseMessage,HumanMessage
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import MemorySaver

In [26]:
load_dotenv()
if not os.getenv("GROQ_API_KEY"): raise ValueError("GROQ_API_KEY not found in .env")
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.1,
    max_tokens=128
)

In [28]:
from langgraph.graph.message import add_messages
class ChatState(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]

In [29]:


def chat_node(state:ChatState):
    messages=state['messages']
    respone=llm.invoke(messages)
    return {'messages':[respone]}

In [33]:
checkpointer=MemorySaver()
graph=StateGraph(ChatState)
graph.add_node('chat_node',chat_node)
graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)
chatbot=graph.compile(checkpointer=checkpointer)

In [32]:
initial_state = {
    "messages": [
        HumanMessage(content="What is the capital of India?")
    ]
}

result = chatbot.invoke(initial_state)
print(result["messages"][-1].content)

The capital of India is **New Delhi**.


In [38]:
thread_id = "1"

config = {
    "configurable": {
        "thread_id": thread_id
    }
}

# 
while True:
    user_input = input("You: ")

    if user_input.strip().lower() in ["exit", "quit"]:
        print("Chat ended.")
        break

    response = chatbot.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        config=config
    )

    print("Bot:", response["messages"][-1].content)

Chat ended.
